<a id="top"></a>

# Lab L0705: wire a tool to a model call

```yaml
title: "Lab L0705: wire a tool to a model call"
keywords: tool calling, bind_tools, tool_calls, ToolMessage, dispatch, round-trip, lab
estimated duration: 35
```

> **Lesson:** L07. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L07/objectives.md). Solutions: `L0705_lab_solutions.ipynb`. Makes **live** Claude calls — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running.
>
> **The client is LangChain's `ChatAnthropic`** (from L03). `bind_tools` turns the plain `calculator` function into a tool; the model's `AIMessage.tool_calls` is the request, and you hand the result back as a `ToolMessage`. See the L0703–L0704 demos.
>
> **After this lab you can:** bind a tool to a model, dispatch the returned tool call to a real function, and hand the result back as a `ToolMessage` to get a final answer.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Send the first turn and find the tool call](#2-problem-1--send-the-first-turn-and-find-the-tool-call)
- [3. Problem 2 — Dispatch: run the real function](#3-problem-2--dispatch-run-the-real-function)
- [4. Problem 3 — Continue: hand the result back](#4-problem-3--continue-hand-the-result-back)
- [5. Problem 4 — Why re-send the tool definition? (written)](#5-problem-4--why-re-send-the-tool-definition-written)
- [6. Self-check](#6-self-check)

## 1. Setup

The LangChain `ChatAnthropic` client, the single `calculator` tool (given), that tool **bound** to the model, and a prompt.

In [ ]:
from typing import Annotated

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage

from fluffy_potato_curriculum.common.config import require_anthropic_key

SONNET = "claude-sonnet-4-6"


# bind_tools infers this tool's definition (name, description, arg schema) from the
# function's name, docstring, and type hint -- a plain typed function IS the schema.
def calculator(
    expression: Annotated[str, "The arithmetic expression to evaluate, e.g. '18374 * 92431'."],
) -> str:
    """Evaluate a basic arithmetic expression (the four operators and parentheses) and
    return the exact numeric result. Use this whenever the user asks for a calculation.

    Raises ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)
model_with_tools = model.bind_tools([calculator])

PROMPT = "What is 6,150 multiplied by 7,012?"
print("setup ready (live model:", SONNET, ")")

[↑ Back to top](#top)

## 2. Problem 1 — Send the first turn and find the tool call

Send `PROMPT` to `model_with_tools` with `await ... .ainvoke(...)`. From the returned `AIMessage`, pull out the first entry of `.tool_calls` (each is a dict with `name`, `args`, and `id`). Print its `name`, `args`, and `id`.

In [ ]:
messages: list[BaseMessage] = [HumanMessage(PROMPT)]
first = await model_with_tools.ainvoke(messages)
call = first.tool_calls[0]
print("name:", call["name"], "| args:", call["args"], "| id:", call["id"])

[↑ Back to top](#top)

## 3. Problem 2 — Dispatch: run the real function

Check the tool name is `calculator`, pull the `expression` argument from `call["args"]`, call `calculator(...)`, and capture the result string. This is the application doing the work the model only *proposed*.

In [ ]:
assert call["name"] == "calculator"
result = calculator(call["args"]["expression"])
print("calculator returned:", result)

[↑ Back to top](#top)

## 4. Problem 3 — Continue: hand the result back

Append the assistant's tool-requesting turn (`first`), then a **`ToolMessage`** whose `tool_call_id` is `call["id"]` and whose `content` is your `result`. `await model_with_tools.ainvoke(...)` on the full conversation again and print the reply's `.content`.

In [ ]:
messages.append(first)
messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
final = await model_with_tools.ainvoke(messages)
print("final answer:", final.content)

[↑ Back to top](#top)

## 5. Problem 4 — Why re-send the tool definition? (written)

In Problem 3 you invoked `model_with_tools` a second time — and because it carries the bound tool, the definition was sent to the model again. In a sentence or two: why is the definition required on every request, not just the first?

_Write your answer by editing this cell (double-click)._

The model is **stateless** across API calls — it keeps no memory of earlier requests. Each call is answered purely from the messages and tool definitions in *that* request, so if the definition isn't re-sent, the model has no record the tool exists and can neither call it nor make sense of the `ToolMessage` that answers a call. `bind_tools` handles this for you by re-attaching the definition to every `ainvoke`.

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0705_lab_solutions.ipynb`. Done when the model proposes a `calculator` call, your dispatch returns `43123800`, and the final answer uses that number. You can state why the tool definition rides along on every call (the model is stateless).

[↑ Back to top](#top)